In [8]:
code = '''"""
Zonal TFT retrain — lr 3e-4 + plateau scheduler.
Base to beat: floor checkpoint (lr=1e-3) = 3.37% zonal MAPE.
"""
from pathlib import Path
import lightning.pytorch as pl
import pandas as pd
import torch
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

FEATURES = Path("zonal_features.parquet")
CKPT_DIR = Path("checkpoints_zonal")
TRAIN_START = "2015-07-01 04:00"
VAL_START   = "2023-01-01 05:00"
TEST_START  = "2024-01-23 12:00"
ENCODER_LEN, DECODER_LEN = 168, 120
BATCH = 128; MAX_EPOCHS = 40; SEED = 42
WEATHER = ["temperature_2m","apparent_temperature","relative_humidity_2m",
           "wind_speed_10m","shortwave_radiation","cloud_cover","temp_vshape"]
UNKNOWN_REALS = ["demand","demand_lag24","demand_lag168",
                 "demand_roll24_mean","demand_roll168_mean","demand_roll24_std"]
KNOWN_REALS = WEATHER + ["time_idx"]
KNOWN_CATS = ["hour","day_of_week","month","is_weekend","is_holiday"]

def main():
    pl.seed_everything(SEED)
    df = pd.read_parquet(FEATURES)
    df["utc"] = pd.to_datetime(df["utc"])
    for c in KNOWN_CATS: df[c] = df[c].astype(str).astype("category")
    df["zone"] = df["zone"].astype(str)
    df = df[df["utc"] >= TRAIN_START].copy()
    df["time_idx"] = df["time_idx"] - df["time_idx"].min()
    val_idx = int(df.loc[df["utc"] >= VAL_START, "time_idx"].min())
    test_idx = int(df.loc[df["utc"] >= TEST_START, "time_idx"].min())
    print(f"boundaries - val:{val_idx} test:{test_idx} max:{df['time_idx'].max()}", flush=True)
    train_df = df[df["time_idx"] < val_idx]
    training_ds = TimeSeriesDataSet(
        train_df, time_idx="time_idx", target="demand", group_ids=["zone"],
        max_encoder_length=ENCODER_LEN, max_prediction_length=DECODER_LEN,
        time_varying_unknown_reals=UNKNOWN_REALS,
        time_varying_known_reals=KNOWN_REALS,
        time_varying_known_categoricals=KNOWN_CATS,
        static_categoricals=["zone"],
        target_normalizer=GroupNormalizer(groups=["zone"]),
        add_relative_time_idx=True, add_target_scales=True,
        allow_missing_timesteps=False,
    )
    validation_ds = TimeSeriesDataSet.from_dataset(
        training_ds, df[df["time_idx"] < test_idx],
        min_prediction_idx=val_idx, stop_randomization=True)
    train_dl = training_ds.to_dataloader(train=True, batch_size=BATCH, num_workers=0)
    val_dl = validation_ds.to_dataloader(train=False, batch_size=BATCH, num_workers=0)
    model = TemporalFusionTransformer.from_dataset(
        training_ds,
        hidden_size=64,
        attention_head_size=4,
        dropout=0.1,
        hidden_continuous_size=32,
        loss=QuantileLoss(),
        learning_rate=3e-4,
        reduce_on_plateau_patience=2,
        log_interval=100,
    )
    print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M", flush=True)
    CKPT_DIR.mkdir(exist_ok=True)
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=6, mode="min"),
        ModelCheckpoint(dirpath=CKPT_DIR, filename="zonal_tft_lr3e4_best",
                        monitor="val_loss", mode="min", save_top_k=1),
        LearningRateMonitor(logging_interval="epoch"),
    ]
    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS, accelerator="auto",
        gradient_clip_val=0.1, callbacks=callbacks,
        enable_progress_bar=False,
    )
    trainer.fit(model, train_dataloaders=train_dl, val_dataloaders=val_dl)
    print(f"\\nBest checkpoint: {callbacks[1].best_model_path}", flush=True)
    print(f"Best val_loss:   {callbacks[1].best_model_score:.4f}", flush=True)

if __name__ == "__main__":
    main()
'''

path = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/04_zonal_tft_train.py"
with open(path, "w") as f:
    f.write(code)
print("written:", path)

written: /opt/app-root/src/Forecasting-Energy-Demand/Sangar/04_zonal_tft_train.py


In [9]:
import subprocess, sys, importlib.util, time

# 1. is anything already training / holding the GPU?
print("GPU processes:")
print(subprocess.run(["nvidia-smi","--query-compute-apps=pid,used_memory","--format=csv"],
                     capture_output=True, text=True).stdout)
print("existing training process:",
      subprocess.run(["pgrep","-af","04_zonal_tft_train"],
                     capture_output=True, text=True).stdout.strip() or "none")

# 2. launch detached
if importlib.util.find_spec("lightning") is None:
    subprocess.run([sys.executable,"-m","pip","install","-q","lightning","pytorch-forecasting"])
    print("packages reinstalled")

log = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/train_lr3e4.log"
with open(log,"w") as f:
    subprocess.Popen([sys.executable,"04_zonal_tft_train.py"],
                     stdout=f, stderr=subprocess.STDOUT, start_new_session=True,
                     cwd="/opt/app-root/src/Forecasting-Energy-Demand/Sangar")
print("\nlaunched detached")
time.sleep(30)
print("--- first 20 log lines ---")
print(subprocess.run(["tail","-20",log], capture_output=True, text=True).stdout)

GPU processes:
pid, used_gpu_memory [MiB]
33885, 716 MiB
211679, 2170 MiB

existing training process: none

launched detached
--- first 20 log lines ---
│ 11 │ lstm_encoder                       │ LSTM      │ 33.3 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM      │ 33.3 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLin… │  8.3 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm   │    128 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedRes… │ 20.9 K │ train │     0 │
│ 16 │ multihead_attn                     │ Interpre… │ 10.4 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddN… │  8.4 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedRes… │ 16.8 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddN… │  8.4 K │ train │     0 │
│ 20 │ output_layer                       │ Linear    │    455 │ train │     0 │
└────┴───────────────────────────────

In [10]:
import os, subprocess
print("this kernel's PID:", os.getpid())
print(subprocess.run(["ps","-o","pid,etime,cmd","-p","211679"],
                     capture_output=True, text=True).stdout)

this kernel's PID: 211679
    PID     ELAPSED CMD
 211679    04:26:12 /opt/app-root/bin/python3 -m ipykernel_launcher -f /opt/app-root/src/.local/share/jupyter/runtime/kernel-b53b1e05-0c85-40d9-a07a-42b16cc0c206.json



In [11]:
!tail -20 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/train_lr3e4.log

│ 11 │ lstm_encoder                       │ LSTM      │ 33.3 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM      │ 33.3 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLin… │  8.3 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm   │    128 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedRes… │ 20.9 K │ train │     0 │
│ 16 │ multihead_attn                     │ Interpre… │ 10.4 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddN… │  8.4 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedRes… │ 16.8 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddN… │  8.4 K │ train │     0 │
│ 20 │ output_layer                       │ Linear    │    455 │ train │     0 │
└────┴────────────────────────────────────┴───────────┴────────┴───────┴───────┘
Trainable params: 403 K                                                         
Non-trainable params: 0     

In [4]:
import subprocess, glob, os
root = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"

print("training alive:",
      subprocess.run(["pgrep","-af","04_zonal_tft_train"], capture_output=True, text=True).stdout.strip() or "DEAD")
print("\n--- last 15 log lines ---")
print(subprocess.run(["tail","-15", f"{root}/train_lr3e4.log"], capture_output=True, text=True).stdout)
print("--- checkpoint written? ---")
print(subprocess.run(["ls","-l", f"{root}/checkpoints_zonal/"], capture_output=True, text=True).stdout)
print("--- newest lightning_logs version ---")
vs = sorted(glob.glob(f"{root}/lightning_logs/version_*"), key=os.path.getmtime)
print(vs[-1] if vs else "none")

training alive: DEAD

--- last 15 log lines ---
│ 16 │ multihead_attn                     │ Interpre… │ 10.4 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddN… │  8.4 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedRes… │ 16.8 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddN… │  8.4 K │ train │     0 │
│ 20 │ output_layer                       │ Linear    │    455 │ train │     0 │
└────┴────────────────────────────────────┴───────────┴────────┴───────┴───────┘
Trainable params: 403 K                                                         
Non-trainable params: 0                                                         
Total params: 403 K                                                             
Total estimated model params size (MB): 1.612                                   
Modules in train mode: 592                                                      
Modules in eval mode: 0                                      

In [5]:
import torch, glob, os, subprocess
root = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"

print("process:", subprocess.run(["pgrep","-af","04_zonal_tft_train"],
                                 capture_output=True, text=True).stdout.strip() or "DEAD")

ck = torch.load(f"{root}/checkpoints_zonal/zonal_tft_lr3e4_best.ckpt",
                map_location="cpu", weights_only=False)
print("epoch:", ck.get("epoch"), "| global_step:", ck.get("global_step"))
for cb, state in ck.get("callbacks", {}).items():
    if "ModelCheckpoint" in str(cb):
        print("best val_loss:", state.get("best_model_score"))
print("lr in ckpt:", ck["hyper_parameters"].get("learning_rate"))

print("\n--- version_32 contents ---")
print(subprocess.run(["ls","-l", f"{root}/lightning_logs/version_32"],
                     capture_output=True, text=True).stdout)

process: DEAD
epoch: 1 | global_step: 11256
best val_loss: tensor(20.8578)
lr in ckpt: 0.0003

--- version_32 contents ---
total 1940
-rw-r--r--. 1 1000950000 1000950000 1942911 Jul 22 05:51 events.out.tfevents.1784688932.dsteam1-0.236823.0
-rw-r--r--. 1 1000950000 1000950000   40866 Jul 22 02:55 hparams.yaml



In [6]:
import subprocess, sys
try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
except ImportError:
    subprocess.run([sys.executable,"-m","pip","install","-q","tensorboard"])
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

p = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/lightning_logs/version_32"
ea = EventAccumulator(p); ea.Reload()
print("available:", ea.Tags()["scalars"])
for tag in ["val_loss", "train_loss_epoch", "lr-Adam"]:
    if tag in ea.Tags()["scalars"]:
        print(f"\n--- {tag} ---")
        for e in ea.Scalars(tag):
            print(f"  step {e.step:>7}  {e.value:.4f}")

available: ['hp_metric', 'lr-Adam', 'train_loss_step', 'epoch', 'val_loss', 'val_SMAPE', 'val_MAE', 'val_RMSE', 'val_MAPE', 'train_loss_epoch']

--- val_loss ---
  step    5627  22.3451
  step   11255  20.8578

--- train_loss_epoch ---
  step    5627  25.7595
  step   11255  18.6622

--- lr-Adam ---
  step       0  0.0003
  step    5628  0.0003
  step   11256  0.0003


In [7]:
code = '''"""
Zonal TFT retrain — lr 3e-4 + plateau scheduler, speed-optimized and resumable.
Base to beat: floor checkpoint (lr=1e-3) = 3.37% zonal MAPE.
"""
from pathlib import Path
import lightning.pytorch as pl
import pandas as pd
import torch
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

FEATURES = Path("zonal_features.parquet")
CKPT_DIR = Path("checkpoints_zonal")
TRAIN_START = "2015-07-01 04:00"
VAL_START   = "2023-01-01 05:00"
TEST_START  = "2024-01-23 12:00"
ENCODER_LEN, DECODER_LEN = 168, 120
BATCH = 128; MAX_EPOCHS = 40; SEED = 42
NUM_WORKERS = 4

WEATHER = ["temperature_2m","apparent_temperature","relative_humidity_2m",
           "wind_speed_10m","shortwave_radiation","cloud_cover","temp_vshape"]
UNKNOWN_REALS = ["demand","demand_lag24","demand_lag168",
                 "demand_roll24_mean","demand_roll168_mean","demand_roll24_std"]
KNOWN_REALS = WEATHER + ["time_idx"]
KNOWN_CATS = ["hour","day_of_week","month","is_weekend","is_holiday"]

def main():
    pl.seed_everything(SEED)
    torch.set_float32_matmul_precision("high")

    df = pd.read_parquet(FEATURES)
    df["utc"] = pd.to_datetime(df["utc"])
    for c in KNOWN_CATS: df[c] = df[c].astype(str).astype("category")
    df["zone"] = df["zone"].astype(str)
    df = df[df["utc"] >= TRAIN_START].copy()
    df["time_idx"] = df["time_idx"] - df["time_idx"].min()
    val_idx = int(df.loc[df["utc"] >= VAL_START, "time_idx"].min())
    test_idx = int(df.loc[df["utc"] >= TEST_START, "time_idx"].min())
    max_idx = int(df["time_idx"].max())
    print(f"boundaries - val:{val_idx} test:{test_idx} max:{max_idx}", flush=True)

    train_df = df[df["time_idx"] < val_idx]
    training_ds = TimeSeriesDataSet(
        train_df, time_idx="time_idx", target="demand", group_ids=["zone"],
        max_encoder_length=ENCODER_LEN, max_prediction_length=DECODER_LEN,
        time_varying_unknown_reals=UNKNOWN_REALS,
        time_varying_known_reals=KNOWN_REALS,
        time_varying_known_categoricals=KNOWN_CATS,
        static_categoricals=["zone"],
        target_normalizer=GroupNormalizer(groups=["zone"]),
        add_relative_time_idx=True, add_target_scales=True,
        allow_missing_timesteps=False,
    )
    validation_ds = TimeSeriesDataSet.from_dataset(
        training_ds, df[df["time_idx"] < test_idx],
        min_prediction_idx=val_idx, stop_randomization=True)

    train_dl = training_ds.to_dataloader(train=True, batch_size=BATCH,
                                         num_workers=NUM_WORKERS, persistent_workers=True)
    val_dl = validation_ds.to_dataloader(train=False, batch_size=BATCH,
                                         num_workers=NUM_WORKERS, persistent_workers=True)

    model = TemporalFusionTransformer.from_dataset(
        training_ds,
        hidden_size=64,
        attention_head_size=4,
        dropout=0.1,
        hidden_continuous_size=32,
        loss=QuantileLoss(),
        learning_rate=3e-4,
        reduce_on_plateau_patience=2,
        log_interval=100,
    )
    print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M", flush=True)

    CKPT_DIR.mkdir(exist_ok=True)
    ckpt_cb = ModelCheckpoint(dirpath=CKPT_DIR, filename="zonal_tft_lr3e4_best",
                              monitor="val_loss", mode="min", save_top_k=1,
                              save_last=True)
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=6, mode="min"),
        ckpt_cb,
        LearningRateMonitor(logging_interval="epoch"),
    ]
    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS, accelerator="auto",
        precision="16-mixed",
        gradient_clip_val=0.1, callbacks=callbacks,
        enable_progress_bar=False,
        log_every_n_steps=200,
    )

    resume = CKPT_DIR / "last.ckpt"
    resume_path = str(resume) if resume.exists() else None
    print(f"resuming from: {resume_path}", flush=True)

    trainer.fit(model, train_dataloaders=train_dl, val_dataloaders=val_dl,
                ckpt_path=resume_path)

    print("Best checkpoint:", ckpt_cb.best_model_path, flush=True)
    print("Best val_loss:  ", float(ckpt_cb.best_model_score), flush=True)

if __name__ == "__main__":
    main()
'''

path = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/04_zonal_tft_train.py"
with open(path, "w") as f:
    f.write(code)
print("written:", path)

written: /opt/app-root/src/Forecasting-Energy-Demand/Sangar/04_zonal_tft_train.py


In [11]:
import subprocess, sys, time

root = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
path = f"{root}/04_zonal_tft_train.py"

src = open(path).read()
src = src.replace('        precision="bf16-mixed",\n', '')
open(path, "w").write(src)
print("precision removed:", "precision=" not in src)

log = f"{root}/train_lr3e4.log"
with open(log, "w") as f:
    subprocess.Popen([sys.executable, "04_zonal_tft_train.py"],
                     stdout=f, stderr=subprocess.STDOUT, start_new_session=True, cwd=root)
print("launched")
time.sleep(90)
print(subprocess.run(["tail", "-6", log], capture_output=True, text=True).stdout)

precision removed: True
launched
Non-trainable params: 0                                                         
Total params: 403 K                                                             
Total estimated model params size (MB): 1.612                                   
Modules in train mode: 592                                                      
Modules in eval mode: 0                                                         
Total FLOPs: 0                                                                  



In [13]:
import subprocess, os
root = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
print("alive:", subprocess.run(["pgrep","-f","04_zonal_tft_train"], capture_output=True, text=True).stdout.strip() or "DEAD")
print(subprocess.run(["ls","-lt", f"{root}/checkpoints_zonal/"], capture_output=True, text=True).stdout)

alive: DEAD
total 26880
-rw-r--r--. 1 1000950000 1000950000 5503858 Jul 22 08:43 last.ckpt
-rw-r--r--. 1 1000950000 1000950000 5503858 Jul 22 08:43 zonal_tft_lr3e4_best-v1.ckpt
-rw-r--r--. 1 1000950000 1000950000 5503794 Jul 22 05:09 zonal_tft_lr3e4_best.ckpt
-rw-r--r--. 1 1000950000 1000950000 5503794 Jul 22 05:09 zonal_tft_lr3e4_epoch1_backup.ckpt
-rw-rw-r--. 1 1000950000 1000950000 5503794 Jul 12 22:44 zonal_tft_best.ckpt



In [14]:
import subprocess, sys, time
root = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
log = f"{root}/train_lr3e4.log"
with open(log, "w") as f:
    subprocess.Popen([sys.executable, "04_zonal_tft_train.py"],
                     stdout=f, stderr=subprocess.STDOUT, start_new_session=True, cwd=root)
print("relaunched")
time.sleep(60)
print(subprocess.run(["tail", "-4", log], capture_output=True, text=True).stdout)

relaunched
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL pytorch_forecasting.data.encoders.GroupNormalizer was not an allowed global by default. Please use `torch.serialization.add_safe_globals([pytorch_forecasting.data.encoders.GroupNormalizer])` or the `torch.serialization.safe_globals([pytorch_forecasting.data.encoders.GroupNormalizer])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.



In [16]:
import subprocess, sys, time

root = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
path = f"{root}/04_zonal_tft_train.py"

src = open(path).read()
src = src.replace(
    "trainer.fit(model, train_dataloaders=train_dl, val_dataloaders=val_dl,\n                ckpt_path=resume_path)",
    "trainer.fit(model, train_dataloaders=train_dl, val_dataloaders=val_dl,\n                ckpt_path=resume_path, weights_only=False)"
)
open(path, "w").write(src)
print("patched:", "weights_only=False" in src)

log = f"{root}/train_lr3e4.log"
with open(log, "w") as f:
    subprocess.Popen([sys.executable, "04_zonal_tft_train.py"],
                     stdout=f, stderr=subprocess.STDOUT, start_new_session=True, cwd=root)
print("relaunched")
time.sleep(90)
print(subprocess.run(["grep", "-E", "resuming|Error|error", log], capture_output=True, text=True).stdout)
print(subprocess.run(["tail", "-4", log], capture_output=True, text=True).stdout)

patched: True
relaunched
resuming from: checkpoints_zonal/last.ckpt

Modules in train mode: 592                                                      
Modules in eval mode: 0                                                         
Total FLOPs: 0                                                                  
Restored all states from the checkpoint at checkpoints_zonal/last.ckpt



In [17]:
import subprocess
root = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"

print("process:", subprocess.run(["pgrep","-f","04_zonal_tft_train"],
      capture_output=True, text=True).stdout.strip() or "DEAD")
print("\n--- checkpoints (newest first) ---")
print(subprocess.run(["ls","-lt", f"{root}/checkpoints_zonal/"],
      capture_output=True, text=True).stdout)
print("--- val_loss curve so far ---")
try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    import glob, os
    v = sorted(glob.glob(f"{root}/lightning_logs/version_*"), key=os.path.getmtime)[-1]
    ea = EventAccumulator(v); ea.Reload()
    if "val_loss" in ea.Tags()["scalars"]:
        for e in ea.Scalars("val_loss"):
            print(f"  step {e.step:>7}  val_loss {e.value:.4f}")
    else:
        print("  no val_loss logged yet in", v)
except Exception as ex:
    print("  tensorboard read failed:", ex)

process: 3338
3597
3598
3599
3600
3613
3614
3615
3616
4142
4400
4401
4402
4403
4417
4418
4419
4420

--- checkpoints (newest first) ---
total 26912
-rw-r--r--. 1 1000950000 1000950000 5518514 Jul 22 13:34 last.ckpt
-rw-r--r--. 1 1000950000 1000950000 5518514 Jul 22 13:34 zonal_tft_lr3e4_best-v1.ckpt
-rw-r--r--. 1 1000950000 1000950000 5503794 Jul 22 05:09 zonal_tft_lr3e4_best.ckpt
-rw-r--r--. 1 1000950000 1000950000 5503794 Jul 22 05:09 zonal_tft_lr3e4_epoch1_backup.ckpt
-rw-rw-r--. 1 1000950000 1000950000 5503794 Jul 12 22:44 zonal_tft_best.ckpt

--- val_loss curve so far ---
  no val_loss logged yet in /opt/app-root/src/Forecasting-Energy-Demand/Sangar/lightning_logs/version_39


In [18]:
import glob, os
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

root = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
vs = sorted(glob.glob(f"{root}/lightning_logs/version_*"), key=os.path.getmtime)[-5:]
for v in vs:
    ea = EventAccumulator(v); ea.Reload()
    if "val_loss" in ea.Tags()["scalars"]:
        pts = ea.Scalars("val_loss")
        print(os.path.basename(v), "->", ", ".join(f"{e.value:.3f}" for e in pts))
    else:
        print(os.path.basename(v), "-> no val_loss yet")

version_35 -> no val_loss yet
version_36 -> 22.671, 21.538, 21.274, 21.395, 21.750, 21.955, 22.862, 22.505, 23.167
version_37 -> no val_loss yet
version_38 -> 21.200, 21.518, 22.704
version_39 -> no val_loss yet
